# AlphaFold DB Fetch — Example

This notebook demonstrates `run_alphafold_db_fetch`, which retrieves an AlphaFold prediction record from the [AlphaFold Protein Structure Database](https://alphafold.ebi.ac.uk/) by UniProt accession. It returns the predicted structure (PDB or mmCIF), per-residue pLDDT confidence scores, and optionally the predicted aligned error (PAE) matrix, alongside metadata such as gene name, organism, model version, and source URLs. See the AlphaFold DB paper: Varadi et al., *Nucleic Acids Research* (2022), [doi:10.1093/nar/gkab1061](https://doi.org/10.1093/nar/gkab1061).

In [1]:
from proto_tools.tools.database_retrieval import (
    AlphaFoldDBFetchConfig,
    AlphaFoldDBFetchInput,
    run_alphafold_db_fetch,
)

## Input API Reference

| Field | Type | Required | Description |
|---|---|---|---|
| `uniprot_id` | `str` | yes | UniProt accession (e.g. `'P04637'`) |

## Config API Reference

| Field | Type | Default | Description |
|---|---|---|---|
| `structure_format` | `Literal["pdb", "cif"]` | `"pdb"` | Structure file format to download |
| `include_structure` | `bool` | `True` | Download the structure file text into the output |
| `include_plddt` | `bool` | `True` | Download the per-residue pLDDT confidence array |
| `include_pae` | `bool` | `False` | Download the PAE matrix (large for long proteins) |
| `request_timeout_seconds` | `int` | `15` | HTTP timeout in seconds |
| `http_retries` | `int` | `2` | Retries for failed HTTP requests |
| `backoff_seconds` | `float` | `1.0` | Seconds to wait between retries (doubles each attempt) |
| `user_agent` | `str` | `"proto-tools/alphafold-db-fetch-v1"` | Identifier sent to the AlphaFold DB API |

## Output API Reference

| Field | Type | Description |
|---|---|---|
| `uniprot_accession` | `str` | Primary UniProt accession that was looked up |
| `entry_id` | `str` | AlphaFold entry identifier (e.g. `'AF-P04637-F1'`) |
| `sequence` | `str` | Amino-acid sequence covered by the prediction |
| `sequence_length` | `int` | Length of the predicted sequence |
| `latest_version` | `int` | Latest available model version on AlphaFold DB |
| `mean_plddt` | `float \| None` | Mean per-residue pLDDT for the prediction |
| `pdb_url` | `str` | URL to the PDB structure file on AlphaFold DB |
| `cif_url` | `str` | URL to the mmCIF structure file on AlphaFold DB |
| `plddt_doc_url` | `str` | URL to the per-residue pLDDT JSON document |
| `pae_doc_url` | `str` | URL to the PAE JSON document |
| `structure_format` | `str` | Format of the downloaded structure (`'pdb'` or `'cif'`); empty when not requested |
| `structure_text` | `str \| None` | Structure file contents in `structure_format` (None if not requested) |
| `plddt_per_residue` | `list[float] \| None` | Per-residue pLDDT scores (0–100); None if not requested |
| `pae_matrix` | `list[list[float]] \| None` | N×N predicted aligned error matrix in angstroms; None if not requested |
| `raw_entry` | `dict[str, Any]` | Complete AlphaFold DB JSON record for advanced programmatic access |

In [2]:
# Basic usage: fetch the AlphaFold prediction for human TP53 (UniProt P04637)
# with all defaults (PDB structure + per-residue pLDDT, PAE off).
inputs = AlphaFoldDBFetchInput(uniprot_id="P04637")
config = AlphaFoldDBFetchConfig()

output = run_alphafold_db_fetch(inputs, config)

print(f"UniProt accession: {output.uniprot_accession}")
print(f"AlphaFold entry id: {output.entry_id}")
print(f"Sequence length:   {output.sequence_length}")
print(f"Mean pLDDT:        {output.mean_plddt}")

print("\nStructure file (first 80 chars):")
print(output.structure_text[:80])

print(f"\nper-residue pLDDT length: {len(output.plddt_per_residue)}")
print(f"first 5 pLDDT values:     {output.plddt_per_residue[:5]}")

UniProt accession: P04637
AlphaFold entry id: AF-P04637-F1
Sequence length:   393
Mean pLDDT:        75.06

Structure file (first 80 chars):
HEADER                                            01-AUG-25                     

per-residue pLDDT length: 393
first 5 pLDDT values:     [42.72, 43.59, 37.78, 45.91, 39.31]


In [3]:
# With the PAE matrix: useful for assessing inter-domain confidence.
# PAE files can be large for long proteins, so this is opt-in.
pae_output = run_alphafold_db_fetch(
    AlphaFoldDBFetchInput(uniprot_id="P04637"),
    AlphaFoldDBFetchConfig(include_pae=True),
)

n_rows = len(pae_output.pae_matrix)
n_cols = len(pae_output.pae_matrix[0])
print(f"PAE matrix shape: ({n_rows}, {n_cols})")

print("\nTop-left 3x3 corner (angstroms):")
for row in pae_output.pae_matrix[:3]:
    print([round(v, 3) for v in row[:3]])

PAE matrix shape: (393, 393)

Top-left 3x3 corner (angstroms):
[0.0, 1.0, 3.0]
[2.0, 0.0, 1.0]
[3.0, 2.0, 0.0]


In [4]:
# Metadata-only fetch: skip structure, pLDDT, and PAE downloads.
# This is the cheapest call when you only need URLs and identifiers.
meta_output = run_alphafold_db_fetch(
    AlphaFoldDBFetchInput(uniprot_id="P04637"),
    AlphaFoldDBFetchConfig(
        include_structure=False,
        include_plddt=False,
        include_pae=False,
    ),
)

print(f"PDB URL:        {meta_output.pdb_url}")
print(f"mmCIF URL:      {meta_output.cif_url}")
print(f"pLDDT doc URL:  {meta_output.plddt_doc_url}")
print(f"PAE doc URL:    {meta_output.pae_doc_url}")

PDB URL:        https://alphafold.ebi.ac.uk/files/AF-P04637-F1-model_v6.pdb
mmCIF URL:      https://alphafold.ebi.ac.uk/files/AF-P04637-F1-model_v6.cif
pLDDT doc URL:  https://alphafold.ebi.ac.uk/files/AF-P04637-F1-confidence_v6.json
PAE doc URL:    https://alphafold.ebi.ac.uk/files/AF-P04637-F1-predicted_aligned_error_v6.json


In [5]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("alphafold_db_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'alphafold_db_results.json'}")

Exported to example_output/alphafold_db_results.json
